# Song Popularity Prediction

In this notebook, we will work with the processed file created and train some model on it.<br><br>

**1. Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

**2. Loading the dataset**

In [2]:
df = pd.read_csv('../data/processed/processed_spotify_dataset.csv')  
df.sample(5)

,popularity,duration_ms,explicit,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,...,spanish,study,swedish,synth-pop,tango,techno,trance,trip-hop,turkish,world-music
109819,47,0.290702,0,-0.058345,5,-1.098482,1,-0.316826,-0.558003,1.935086,...,0,0,0,0,0,1,0,0,0,0
86686,38,-0.165250,0,1.431270,9,0.373282,1,0.819423,-1.139596,-0.496536,...,0,0,0,0,0,0,0,0,0,0
51415,0,-0.536384,1,-0.454651,8,0.299582,1,1.002037,-0.978185,-0.593671,...,0,0,0,0,0,0,0,0,0,0
28536,29,-0.011601,0,-0.255535,2,0.061946,1,-0.569760,0.982285,-0.593638,...,0,0,0,0,0,0,0,0,0,0
13135,15,0.525655,0,0.983946,9,-0.954132,1,-0.537656,-1.150776,1.934965,...,0,0,0,0,0,0,0,0,0,0


**3. Splitting Train - Test dataset and Splitting target feature**

In [3]:
X = df.drop(columns=['popularity'])
y = df['popularity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**4. Defining function to predict and evaluate for each ensemble model**

In [4]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    return mae, rmse, r2


models = {
    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ),
    
    "LightGBM": LGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=-1,
        random_state=42
    ),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
}

In [5]:
results = []

for name, model in models.items():
    mae, rmse, r2 = evaluate_model(model, X_train, X_test, y_train, y_test)
    
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_R2_Mean": np.mean(cv_scores)
    })

results_df = pd.DataFrame(results)
print(results_df)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006545 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3700
[LightGBM] [Info] Number of data points in the train set: 91199, number of used features: 135
[LightGBM] [Info] Start training from score 33.298644
           Model       MAE       RMSE        R2  CV_R2_Mean
0        XGBoost  5.893753  11.368983  0.739455    0.713173
1       LightGBM  5.837885  11.288164  0.743146    0.717857
2  Random Forest  5.841065  11.343247  0.740633    0.708050


**5. Hyperparameter Tuning of the models**

In [6]:
lgb = LGBMRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [20, 31, 50],
    'max_depth': [-1, 5, 10],
    'min_child_samples': [10, 20, 30]
}

grid = GridSearchCV(
    lgb,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
best_lgb = grid.best_estimator_

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035359 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3700
[LightGBM] [Info] Number of data points in the train set: 91199, number of used features: 135
[LightGBM] [Info] Start training from score 33.298644
Best Params: {'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 30, 'n_estimators': 500, 'num_leaves': 50}


In [13]:
lgbm = LGBMRegressor(**grid.best_params_, random_state=42)
mae, rmse, r2 = evaluate_model(lgbm, X_train, X_test, y_train, y_test)

cv_lgbm = cross_val_score(lgbm, X, y, cv=5, scoring='r2', n_jobs=-1)

print(f"LightGBM - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
print(f"LightGBM CV R2 Mean: {np.mean(cv_lgbm):.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006190 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3700
[LightGBM] [Info] Number of data points in the train set: 91199, number of used features: 135
[LightGBM] [Info] Start training from score 33.298644
LightGBM - MAE: 5.6227, RMSE: 10.7161, R2: 0.7685
LightGBM CV R2 Mean: 0.7397


In [8]:
xgb = XGBRegressor(random_state=42, verbosity=0)

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

grid_xgb = GridSearchCV(
    xgb,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("Best XGB Params:", grid_xgb.best_params_)
best_xgb = grid_xgb.best_estimator_

Best XGB Params: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 500, 'subsample': 1.0}


In [14]:
xgbm = XGBRegressor(**grid_xgb.best_params_, random_state=42)
mae, rmse, r2 = evaluate_model(xgbm, X_train, X_test, y_train, y_test)

print(f"XGBoost - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

cv_xgb = cross_val_score(xgbm, X, y, cv=5, scoring='r2', n_jobs=-1)

print(f"XGBoost CV R2 Mean: {np.mean(cv_xgb):.4f}")

XGBoost - MAE: 5.7677, RMSE: 10.9696, R2: 0.7574
XGBoost CV R2 Mean: 0.7300


The best result obtained so far is from *LGBM* that is 0.768 using the best grid parameters. The mean cv score is 0.739